# TweetQA: reproduce the selected validation system

**Your task:** independently check your teammates' saved system, not train a new model.

Choose **Runtime → Change runtime type → T4 GPU** and run cells in order. Upload only the provided `validation_inputs.zip` when prompted. It contains the exact source snapshot, training CSV, selected model archives, and reference predictions. It contains **no test CSV**.

The selected predictor averages three v2 models' start/end probabilities, proposes ten legal spans, and applies the original 30-feature OOF LambdaMART reranker. Success requires **1,076 / 1,076 exact answer-string matches**, the same row IDs, and all five metrics matching within 0.0001.

This notebook has been assembled and statically checked locally. The full model reproduction must still be run in your clean Colab runtime. No success is assumed in advance.


## Step 1 — Prepare the runtime

**Why:** Colab supplies PyTorch. Install the other libraries used by your teammates. No exact original environment lockfile was saved, so we record the installed versions and verify behaviour against saved answers.


In [1]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "gensim", "emoji", "nltk", "scikit-learn", "joblib", "lightgbm"])
import torch
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "unavailable (CPU inference is possible but slower)")


PyTorch: 2.11.0+cu130
GPU: Tesla T4


## Step 2 — Upload and verify the inputs

**What to do:** select `validation_inputs.zip` from the files supplied with this notebook.

**Why:** hashes check that we use the exact saved files. We extract into a fresh temporary folder, so reruns do not mix old results or imports. Never upload `test.csv`.


In [2]:
from google.colab import files
from pathlib import Path
import hashlib, json, zipfile, tempfile, os
uploaded = files.upload()
assert len(uploaded) == 1 and "validation_inputs.zip" in uploaded, "Upload only validation_inputs.zip"
bundle_bytes = uploaded["validation_inputs.zip"]
assert hashlib.sha256(bundle_bytes).hexdigest() == '58b0e8e021c224e92c0f526af7e8e7c1558ce0ca11609f6a01c553cb7d8815f5', "Input bundle identity differs"
WORK = Path(tempfile.mkdtemp(prefix="tweetqa_validation_", dir="/content"))
import io
with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as z:
    for name in z.namelist():
        p = Path(name)
        assert not p.is_absolute() and ".." not in p.parts, name
        assert p.name != "test.csv", "Test data must not be included"
    z.extractall(WORK)
del uploaded, bundle_bytes
os.chdir(WORK)
sha256 = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
bundle_manifest = json.loads(Path("bundle_manifest.json").read_text())
for name, expected in bundle_manifest["file_sha256"].items():
    assert sha256(name) == expected, f"File identity differs: {name}"
assert not Path("data/test.csv").exists()
assert hashlib.sha1(Path("data/train.csv").read_bytes().replace(b"\r\n", b"\n")).hexdigest() == "8a52bb7dff56b11fdad1ab6fd3b848fbadc22849"
print("PASS: exact input bundle verified; test data absent")
print("Source commit:", bundle_manifest["git_commit"])
print("Working folder:", WORK)


Saving validation_inputs.zip to validation_inputs.zip
PASS: exact input bundle verified; test data absent
Source commit: 2a5fb8d14be5af8d009d1f8eea8ca6be7db72f0d
Working folder: /content/tweetqa_validation_8fbdv6g3


## Step 3 — Load only the selected saved models

**Why:** similarly named files belong to different experiments. Extract the three selected QA checkpoints and the original reranker only. Loading models does not train them. These trusted team artifacts contain Python-serialized objects.


In [3]:
CHECKPOINT_HASHES = {'checkpoints/A5_single_span-s1-20261014-071243.pt': '1761eba471347e22b2f5313a5d4e9f021d1c571baae2a1ff6909ead65ff80e36', 'checkpoints/A5_single_span-s2-20261014-071828.pt': '7a75861188cf954daa8a8ca587a59c77636f71e316ea72092f84ff23e5ed63ec', 'checkpoints/A5_single_span-s3-20261014-072428.pt': 'c81dde15edf4679d2d3dda90ef2d9108ab837d449e855435336e163f07b128d9'}
ARCHIVE_HASHES = {'a2_v2_colab_artifacts.zip': '6fc46dc5ef256426d8c4b95da2898483edba34687a62013bc2889cd59c9badd4', 'oof_span_reranker_artifacts.zip': '83436c4eff8604c8ebed3f898fab5874e5e67972097743b13389e4fcd05e4c00', 'ensemble_feature_artifacts.zip': 'a257c825adab25438dd98899ca3178c83eb141e5c6a22d5dd3297312f86fccdd'}
RERANKER_MEMBER = 'oof_reranker_results/oof_lambdamart_reranker.joblib'
RERANKER_HASH = '93e3689c180815e972d5efd52fb5c7e0a626aa6261f0739d406c5fcea8ee9d32'
for name, expected in ARCHIVE_HASHES.items():
    assert sha256(name) == expected, name
    with zipfile.ZipFile(name) as z:
        assert z.testzip() is None, f"Damaged archive: {name}"
with zipfile.ZipFile("a2_v2_colab_artifacts.zip") as z:
    for member, expected in CHECKPOINT_HASHES.items():
        payload = z.read(member)
        assert hashlib.sha256(payload).hexdigest() == expected, member
        p = Path(member); p.parent.mkdir(parents=True, exist_ok=True); p.write_bytes(payload)
with zipfile.ZipFile("oof_span_reranker_artifacts.zip") as z:
    payload = z.read(RERANKER_MEMBER)
    assert hashlib.sha256(payload).hexdigest() == RERANKER_HASH
    p = Path(RERANKER_MEMBER); p.parent.mkdir(parents=True, exist_ok=True); p.write_bytes(payload)
# Import only this fresh snapshot; discard previously imported src modules on a rerun.
for name in list(sys.modules):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]
sys.path.insert(0, str(WORK))
os.environ["TWEETQA_DIR"] = str(WORK / "data")
os.environ["A2_RESULTS"] = str(WORK / "reproduction_outputs")
from src import data as D, train as T, rerank as R
from src.evaluate import evaluate
import numpy as np, joblib
assert Path(D.__file__).resolve() == WORK / "src/data.py"
assert D.TEST_LOCKED is True
reranker = joblib.load(RERANKER_MEMBER)
assert reranker.n_features_in_ == len(R.FEATURE_NAMES) == 30
assert not hasattr(reranker, "decision_function"), "Expected LambdaMART, not the linear alternative"
checkpoint_paths = [Path(p) for p in CHECKPOINT_HASHES]
device = "cuda" if torch.cuda.is_available() else "cpu"
OUT = WORK / "reproduction_outputs"
OUT.mkdir()
print("PASS: selected checkpoint identities and 30-feature reranker verified")
print("Device:", device)


PASS: selected checkpoint identities and 30-feature reranker verified
Device: cuda


## Step 4 — Recreate the original validation questions

**Why:** reproduce the same split instead of randomly splitting again. Keep all reference answers separate from prediction inputs. Repeated question IDs must not be discarded.


In [4]:
train_rows, val_rows = D.get_splits()
assert D.SPLIT_SEED == 4012 and D.VAL_FRAC == 0.10
assert len(train_rows) == 9570 and len(val_rows) == 1076
assert len({r["tweet"] for r in val_rows}) == 511
assert not ({r["tweet"] for r in train_rows} & {r["tweet"] for r in val_rows})
assert len({r["row_id"] for r in val_rows}) == len(val_rows)
golds = [r["answers"] for r in val_rows]
with zipfile.ZipFile("ensemble_feature_artifacts.zip") as z:
    reference_rows = json.loads(z.read("Ensemble_original_predictions.json"))
    reference_summary = json.loads(z.read("summary.json"))
reference = {r["row_id"]: r["pred"] for r in reference_rows}
assert len(reference) == len(reference_rows) == len(val_rows)
assert set(reference) == {r["row_id"] for r in val_rows}
expected_metrics = reference_summary["scores"]["Ensemble_original"]
print(f"PASS: {len(val_rows)} validation questions from 511 tweets; IDs match reference")
print("Expected scores:", expected_metrics)


PASS: 1076 validation questions from 511 tweets; IDs match reference
Expected scores: {'bleu1': 41.611863196854394, 'meteor': 34.857259256122916, 'rougeL': 47.111843224551954, 'em': 30.111524163568774, 'f1': 47.31876525048259}


## Step 5 — Define the exact prediction procedure

**Why:** each QA checkpoint owns its vocabulary. Predict separately, average probabilities (not logits), enumerate Top-10 spans up to 15 tokens, and let the fixed reranker select one. This function follows section 7 of `FINAL_MODEL_HANDOFF.md`; the only added change is progress printing.

The function replaces gold answers with empty placeholders before preprocessing. All validation rows receive predictions.


In [5]:
def predict_selected_system(raw_rows, checkpoint_paths, reranker, device="cuda"):
    import json
    assert len(checkpoint_paths) == 3
    assert len({r["row_id"] for r in raw_rows}) == len(raw_rows)
    assert reranker.n_features_in_ == len(R.FEATURE_NAMES) == 30
    rows = [dict(r, answers=[""], answer_field="") for r in raw_rows]
    reference_examples = None
    all_probabilities = []
    reference_cfg = None

    def effective(cfg):
        return json.loads(json.dumps(
            {**T.DEFAULT_TRAIN, **T._default_model(), **cfg}, sort_keys=True
        ))

    for path in checkpoint_paths:
        print("Predicting with", path.name, flush=True)
        model, cfg, wv, cv = T.load_checkpoint(path, device)
        if reference_cfg is None:
            reference_cfg = effective(cfg)
        assert effective(cfg) == reference_cfg
        assert cfg["max_span_len"] == 15
        assert not cfg.get("use_joint_span_scorer", False)
        examples = D.preprocess(rows, dict(
            use_normaliser=cfg["use_normaliser"],
            max_span_len=cfg["max_span_len"],
            tau=cfg["tau"], soft_tau=cfg.get("soft_tau"),
        ))
        assert all(e["t_tokens"] and e["q_tokens"] for e in examples)
        D.encode(examples, wv, cv)
        if reference_examples is None:
            reference_examples = examples
        else:
            assert [e["row_id"] for e in examples] == [
                e["row_id"] for e in reference_examples
            ]
            assert [e["t_tokens"] for e in examples] == [
                e["t_tokens"] for e in reference_examples
            ]
            assert [e["q_tokens"] for e in examples] == [
                e["q_tokens"] for e in reference_examples
            ]
        values = []
        model.eval()
        with torch.no_grad():
            for batch in D.make_loader(examples, batch_size=64, train=False):
                out = model(T.to_device(batch, device))
                for b, ex in enumerate(batch["meta"]):
                    n = len(ex["t_tokens"])
                    values.append((
                        ex["row_id"],
                        torch.softmax(out["start_logits"][b, :n], -1).cpu(),
                        torch.softmax(out["end_logits"][b, :n], -1).cpu(),
                    ))
        assert [v[0] for v in values] == [r["row_id"] for r in rows]
        all_probabilities.append(values)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    records = []
    for k, ex in enumerate(reference_examples):
        items = [values[k] for values in all_probabilities]
        assert all(item[0] == ex["row_id"] for item in items)
        start = torch.stack([x[1] for x in items]).mean(0).clamp_min(1e-30).log()
        end = torch.stack([x[2] for x in items]).mean(0).clamp_min(1e-30).log()
        n = len(start)
        positions = torch.arange(n)
        width = positions[None, :] - positions[:, None]
        legal = (width >= 0) & (width < 15)
        scores = (start[:, None] + end[None, :]).masked_fill(~legal, -float("inf"))
        best_scores, indices = scores.flatten().topk(min(10, int(legal.sum())))
        candidates = []
        for score, flat in zip(best_scores.tolist(), indices.tolist()):
            i, j = divmod(flat, n)
            candidates.append(dict(
                start=i, end=j, text=D.span_text(ex, i, j),
                start_logp=float(start[i]), end_logp=float(end[j]),
                base_score=score, region=R.span_region(ex, i, j),
            ))
        records.append(dict(row_id=ex["row_id"], qid=ex["qid"], candidates=candidates))

    predictions, selections = R.rerank(reranker, reference_examples, records)
    assert len(predictions) == len(raw_rows)
    assert all(np.isfinite(s["rerank_score"]) for s in selections)
    outputs = []
    for ex, pred, selection in zip(reference_examples, predictions, selections):
        i, j = selection["start"], selection["end"]
        char_start = ex["t_tokens"][i]["start"]
        char_end = ex["t_tokens"][j]["end"]
        assert pred == ex["tweet"][char_start:char_end]
        outputs.append(dict(
            row_id=ex["row_id"], qid=ex["qid"], prediction=pred,
            selected_rank=selection["selected_rank"],
            start_token=i, end_token=j,
            start_char=char_start, end_char_exclusive=char_end,
            base_score=selection["base_score"],
            rerank_score=selection["rerank_score"],
        ))
    return outputs, records


## Step 6 — Run validation inference and save predictions

**What to expect:** progress prints once for each of the three checkpoints. This runs prediction only, with no training. Wait for the cell to finish before continuing.


In [6]:
from datetime import datetime
from zoneinfo import ZoneInfo
import time, csv
started_at = datetime.now(ZoneInfo("Australia/Perth")).isoformat()
t0 = time.time()
outputs, records = predict_selected_system(val_rows, checkpoint_paths, reranker, device=device)
elapsed_s = time.time() - t0
assert len(outputs) == 1076
assert [r["row_id"] for r in outputs] == [r["row_id"] for r in val_rows]
assert [r["qid"] for r in outputs] == [r["qid"] for r in val_rows]
for raw, result in zip(val_rows, outputs):
    assert result["prediction"] == raw["tweet"][result["start_char"]:result["end_char_exclusive"]]
    assert 0 <= result["start_token"] <= result["end_token"]
    assert result["end_token"] - result["start_token"] < 15
    assert 1 <= result["selected_rank"] <= 10
    assert np.isfinite(result["base_score"]) and np.isfinite(result["rerank_score"])
(OUT / "validation_predictions.json").write_text(json.dumps(outputs, ensure_ascii=False, indent=2))
with (OUT / "validation_predictions.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(outputs[0])); writer.writeheader(); writer.writerows(outputs)
with (OUT / "top10_candidates.jsonl").open("w", encoding="utf-8") as f:
    for record in records: f.write(json.dumps(record, ensure_ascii=False) + "\n")
print(f"Saved {len(outputs)} predictions; inference took {elapsed_s:.1f} seconds")


Predicting with A5_single_span-s1-20261014-071243.pt
Predicting with A5_single_span-s2-20261014-071828.pt
Predicting with A5_single_span-s3-20261014-072428.pt


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:27

Saved 1076 predictions; inference took 11.1 seconds


## Step 7 — Compare exact answers and all five metrics

**Success:** zero mismatches and metric differences below 0.0001. A rounded 47.11 alone is not sufficient.

If this fails, the diagnostic files are saved first. Do not change model settings or tune answers to force agreement. Review the concrete mismatches and runtime differences.


In [7]:
mismatches = [dict(row_id=r["row_id"], expected=reference[r["row_id"]], actual=r["prediction"])
              for r in outputs if r["prediction"] != reference[r["row_id"]]]
(OUT / "answer_mismatches.json").write_text(json.dumps(mismatches, ensure_ascii=False, indent=2))
print(f"Exact answer matches: {len(outputs) - len(mismatches)} / {len(outputs)}")
if mismatches:
    print("First mismatches:", json.dumps(mismatches[:5], ensure_ascii=False, indent=2))
# NLTK downloads WordNet resources on first METEOR evaluation; allow that setup.
scores = evaluate([r["prediction"] for r in outputs], golds)
reference_rescored = evaluate([reference[r["row_id"]] for r in val_rows], golds)
deltas = {m: float(scores[m] - expected_metrics[m]) for m in expected_metrics}
reference_deltas = {m: float(reference_rescored[m] - expected_metrics[m]) for m in expected_metrics}
checks = dict(n_questions=len(outputs), n_tweets=511, exact_matches=len(outputs)-len(mismatches),
              mismatch_count=len(mismatches), scores=scores, expected=expected_metrics,
              metric_deltas=deltas, reference_rescored=reference_rescored,
              reference_metric_deltas=reference_deltas,
              passed=(not mismatches and all(abs(v) < 1e-4 for v in deltas.values())
                      and all(abs(v) < 1e-4 for v in reference_deltas.values())), test_accessed=False)
(OUT / "validation_checks.json").write_text(json.dumps(checks, indent=2))
for metric in expected_metrics:
    print(f"{metric:8s} actual={scores[metric]:.8f} expected={expected_metrics[metric]:.8f} delta={deltas[metric]:+.8f}")
print("PASS: exact validation reproduction" if checks["passed"] else "FAIL: inspect diagnostics; no test access")


Exact answer matches: 1076 / 1076
bleu1    actual=41.61186320 expected=41.61186320 delta=+0.00000000
meteor   actual=34.85725926 expected=34.85725926 delta=+0.00000000
rougeL   actual=47.11184322 expected=47.11184322 delta=+0.00000000
em       actual=30.11152416 expected=30.11152416 delta=+0.00000000
f1       actual=47.31876525 expected=47.31876525 delta=+0.00000000
PASS: exact validation reproduction


## Step 8 — Save and download your verification evidence

**Why:** keep the predictions, comparisons, source identities, and installed versions together. Download this archive even if checks fail. Also use **File → Download → Download .ipynb** to save this notebook with its outputs.

A passed run verifies this validation configuration; it does not establish a test score or authorize test evaluation.


In [8]:
import importlib.metadata, platform
packages = ["torch", "numpy", "gensim", "emoji", "nltk", "scikit-learn", "joblib", "lightgbm"]
versions = {}
for name in packages:
    try: versions[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: versions[name] = "not installed"
run_manifest = dict(started_at=started_at, finished_at=datetime.now(ZoneInfo("Australia/Perth")).isoformat(),
    source=bundle_manifest, bundle_sha256='58b0e8e021c224e92c0f526af7e8e7c1558ce0ca11609f6a01c553cb7d8815f5', python=platform.python_version(),
    versions=versions, device=device,
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    checkpoint_order=[str(p) for p in checkpoint_paths], checkpoint_sha256=CHECKPOINT_HASHES,
    reranker_sha256=RERANKER_HASH, feature_names=list(R.FEATURE_NAMES),
    top_k=10, max_span_len=15, averaging_weights=[1/3]*3, inference_seconds=elapsed_s,
    n_questions=len(outputs), test_accessed=False, test_unlocked=False, validation_passed=checks["passed"])
(OUT / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2))
# Record this ensemble as one system, not another independent seed. This writes only to our fresh output folder.
from src.runlog import log_run
if checks["passed"]:
    log_run("SelectedEnsembleOOFLambdaMART_reproduction", run_manifest, seed="ensemble_1_2_3",
            split="val", scores=scores, notes="Validation reproduction; no training or test access")
assert D.TEST_LOCKED is True and not Path("data/test.csv").exists()
report = ("Validation reproduction: " + ("PASS" if checks["passed"] else "FAIL") + "\n"
          + f"Exact matches: {checks['exact_matches']}/1076\n"
          + f"ROUGE-L: {scores['rougeL']:.8f}\nTest accessed: False\n")
(OUT / "verification_summary.txt").write_text(report)
archive = WORK / "validation_reproduction_results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()): z.write(p, p.name)
print(report)
files.download(str(archive))


Validation reproduction: PASS
Exact matches: 1076/1076
ROUGE-L: 47.11184322
Test accessed: False



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>